### Data Loading

In [12]:
import pandas as pd

buzzfeed_real = pd.read_pickle('data/buzzfeed_real_clean.pkl')
buzzfeed_fake = pd.read_pickle('data/buzzfeed_fake_clean.pkl')

In [13]:
buzzfeed_real.head(1)

,id,title,text,url,top_img,authors,source,publish_date,movies,images,canonical_link,meta_data
0,Fake_1-Webpage,Proof The Mainstream Media Is Manipulating The...,I woke up this morning to find a variation of ...,http://www.addictinginfo.org/2016/09/19/proof-...,http://addictinginfo.addictinginfoent.netdna-c...,Wendy Gittleson,http://www.addictinginfo.org,{'$date': 1474243200000},0,"http://i.imgur.com/JeqZLhj.png,http://addictin...",http://addictinginfo.com/2016/09/19/proof-the-...,"{""publisher"": ""Addicting Info | The Knowledge ..."


In [14]:
buzzfeed_fake.head(1)

,id,title,text,url,top_img,authors,source,publish_date,movies,images,canonical_link,meta_data
0,Fake_1-Webpage,Proof The Mainstream Media Is Manipulating The...,I woke up this morning to find a variation of ...,http://www.addictinginfo.org/2016/09/19/proof-...,http://addictinginfo.addictinginfoent.netdna-c...,Wendy Gittleson,http://www.addictinginfo.org,{'$date': 1474243200000},0,"http://i.imgur.com/JeqZLhj.png,http://addictin...",http://addictinginfo.com/2016/09/19/proof-the-...,"{""publisher"": ""Addicting Info | The Knowledge ..."


### Preprocessing

In [15]:
#function to be applied to article titles and contents for initial text cleaning
import re
import string

def clean_text(text):
    
    #type checking
    if not isinstance(text, str):
        return []

    #lowercasing
    text = text.lower()
    
    #removing punctuation
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\w*\d\w*', '', text)

    text = re.sub('[’‘’“”…]', '', text)
    text = re.sub('\n', '', text)
    text = ' '.join(re.findall(r'\b[a-zA-Z0-9]+\b', text))


    return text

<>:15: SyntaxWarning: invalid escape sequence '\['
<>:17: SyntaxWarning: invalid escape sequence '\w'
<>:15: SyntaxWarning: invalid escape sequence '\['
<>:17: SyntaxWarning: invalid escape sequence '\w'
C:\Users\voldy\AppData\Local\Temp\ipykernel_20964\2203787657.py:15: SyntaxWarning: invalid escape sequence '\['
  text = re.sub('\[.*?\]', '', text)
C:\Users\voldy\AppData\Local\Temp\ipykernel_20964\2203787657.py:17: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)


In [16]:
#fake news preprocessing
buzzfeed_fake['clean_title'] = buzzfeed_fake['title'].apply(clean_text)
buzzfeed_fake['clean_text'] = buzzfeed_fake['text'].apply(clean_text)

#real news preprocessing
buzzfeed_real['clean_title'] = buzzfeed_real['title'].apply(clean_text)
buzzfeed_real['clean_text'] = buzzfeed_real['text'].apply(clean_text)

In [17]:
buzzfeed_fake.head(1)

,id,title,text,url,top_img,authors,source,publish_date,movies,images,canonical_link,meta_data,clean_title,clean_text
0,Fake_1-Webpage,Proof The Mainstream Media Is Manipulating The...,I woke up this morning to find a variation of ...,http://www.addictinginfo.org/2016/09/19/proof-...,http://addictinginfo.addictinginfoent.netdna-c...,Wendy Gittleson,http://www.addictinginfo.org,{'$date': 1474243200000},0,"http://i.imgur.com/JeqZLhj.png,http://addictin...",http://addictinginfo.com/2016/09/19/proof-the-...,"{""publisher"": ""Addicting Info | The Knowledge ...",proof the mainstream media is manipulating the...,i woke up this morning to find a variation of ...


In [18]:
buzzfeed_real.head(1)

,id,title,text,url,top_img,authors,source,publish_date,movies,images,canonical_link,meta_data,clean_title,clean_text
0,Fake_1-Webpage,Proof The Mainstream Media Is Manipulating The...,I woke up this morning to find a variation of ...,http://www.addictinginfo.org/2016/09/19/proof-...,http://addictinginfo.addictinginfoent.netdna-c...,Wendy Gittleson,http://www.addictinginfo.org,{'$date': 1474243200000},0,"http://i.imgur.com/JeqZLhj.png,http://addictin...",http://addictinginfo.com/2016/09/19/proof-the-...,"{""publisher"": ""Addicting Info | The Knowledge ...",proof the mainstream media is manipulating the...,i woke up this morning to find a variation of ...


In [19]:
#set up data for train test split
buzzfeed_fake['real_label'] = 0
buzzfeed_real['real_label'] = 1

#create single dataframe and shuffle it up
full_df = pd.concat([buzzfeed_fake, buzzfeed_real], axis=0, ignore_index = True).sample(frac = 1, random_state = 101).reset_index(drop = True)

### Tokenizing

In [20]:
#create a document term matrix
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words = 'english')
df_cv = cv.fit_transform(full_df['clean_text'])

df_dtm = pd.DataFrame(df_cv.toarray(), columns = cv.get_feature_names_out(), index = full_df.index)

df_dtm

,aaron,abandon,abandoned,abandoning,abandonment,abbreviated,abc,abdullah,abedin,abhorrenthopefully,...,youll,young,younger,youre,youve,yup,zero,zimmer,zipper,zone
0,0,0,0,0,0,0,0,0,1,0,...,0,2,0,0,0,0,0,0,1,0
1,0,0,0,0,0,0,0,0,0,0,...,0,2,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,0,0,0,0,0,0,0,0,0,0,...,0,2,0,0,0,0,0,0,0,0
178,0,0,0,0,0,0,0,0,0,0,...,0,2,0,0,0,0,0,0,0,0
179,0,0,0,0,0,0,0,0,0,0,...,0,2,0,0,0,0,0,0,0,0
180,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [22]:
labelled_dtm = pd.concat([df_dtm, full_df['real_label']], axis = 1)

In [23]:
#save pkl for model training
pd.to_pickle(labelled_dtm, 'data/labelled_dtm.pkl')